# Scraping des annonces immobilières - Phase 1

Ce notebook permet de scraper les annonces immobilières depuis Voursa.com.

## Objectifs
- Collecter ≥ 200 annonces (objectif 500+)
- Extraire tous les champs requis
- Respecter robots.txt et bonnes pratiques
- Exporter vers `data/raw/raw_data.csv`


In [ ]:
# Cellule A - Imports + Config
import sys
import os
from pathlib import Path

# Ajouter le répertoire src au path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / 'src'))

import time
import re
import pandas as pd
from tqdm import tqdm
from src.scraping.common import (
    HEADERS, COLUMNS, clean_text, safe_int, polite_sleep, fetch
)
from src.scraping.voursa import scrape_voursa, parse_listing_page, extract_listing_links

print("✓ Imports réussis")
print(f"✓ Répertoire de travail: {project_root}")


In [ ]:
# Cellule B - Fonction fetch robuste (déjà dans common.py)
# La fonction fetch() est disponible depuis src.scraping.common
# Elle gère les retries automatiquement

# Test de connexion
test_url = "https://voursa.com/FR/categories/real_estate"
try:
    html = fetch(test_url)
    print(f"✓ Connexion réussie à {test_url}")
    print(f"  Taille de la page: {len(html)} caractères")
except Exception as e:
    print(f"✗ Erreur de connexion: {e}")


In [ ]:
# Cellule C - Schéma des colonnes
print("Colonnes du dataset:")
for i, col in enumerate(COLUMNS, 1):
    print(f"  {i:2d}. {col}")

print(f"\nTotal: {len(COLUMNS)} colonnes")


In [ ]:
# Cellule D - Fonction de sauvegarde vers CSV
def save_raw(rows, out_path="data/raw/raw_data.csv"):
    """
    Sauvegarde les données scrapées dans un fichier CSV.
    
    Args:
        rows: Liste de dictionnaires contenant les données
        out_path: Chemin du fichier de sortie
    """
    # Créer le répertoire si nécessaire
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    
    # Créer le DataFrame
    df = pd.DataFrame(rows, columns=COLUMNS)
    
    # Sauvegarder
    df.to_csv(out_path, index=False, encoding='utf-8')
    
    print(f"✓ Sauvegardé: {out_path}")
    print(f"  Lignes: {len(df)}")
    print(f"  Colonnes: {len(df.columns)}")
    
    # Afficher quelques statistiques
    print(f"\nStatistiques:")
    print(f"  Annonces avec prix: {df['prix'].notna().sum()}")
    print(f"  Annonces avec surface: {df['surface_m2'].notna().sum()}")
    print(f"  Annonces avec description: {df['description'].notna().sum()}")
    
    return df

print("✓ Fonction save_raw() définie")


In [ ]:
# Cellule E - Test: Extraction des liens depuis la page liste
print("Test d'extraction des liens d'annonces...")

test_url = "https://voursa.com/FR/categories/real_estate"
html = fetch(test_url)
polite_sleep(2.0)

links = extract_listing_links(html)
print(f"\n✓ {len(links)} liens d'annonces extraits")

# Afficher les 5 premiers liens
print("\nPremiers liens:")
for i, link in enumerate(links[:5], 1):
    print(f"  {i}. {link}")


In [ ]:
# Cellule F - Test: Parsing d'une page annonce
if len(links) > 0:
    test_listing_url = links[0]
    print(f"Test de parsing: {test_listing_url}")
    
    html = fetch(test_listing_url)
    polite_sleep(2.0)
    
    listing_data = parse_listing_page(html, test_listing_url)
    
    print("\n✓ Données extraites:")
    for key, value in listing_data.items():
        if value:
            print(f"  {key}: {str(value)[:100]}")
else:
    print("Aucun lien disponible pour le test")


In [ ]:
# Cellule G - Scraping complet
# Objectif: ≥ 200 annonces (on vise 500+)

MAX_LISTINGS = 500  # Modifier selon besoin

print(f"Démarrage du scraping complet (objectif: {MAX_LISTINGS} annonces)")
print("=" * 60)

all_listings = scrape_voursa(max_listings=MAX_LISTINGS)

print("\n" + "=" * 60)
print(f"✓ Scraping terminé: {len(all_listings)} annonces collectées")


In [ ]:
# Cellule H - Export final + Statistiques
if len(all_listings) > 0:
    # Sauvegarder dans CSV
    df = save_raw(all_listings, "data/raw/raw_data.csv")
    
    # Statistiques détaillées
    print("\n" + "=" * 60)
    print("STATISTIQUES DÉTAILLÉES")
    print("=" * 60)
    
    print(f"\nTotal d'annonces: {len(df)}")
    print(f"\nPar type de bien:")
    if df['type_bien'].notna().any():
        print(df['type_bien'].value_counts())
    
    print(f"\nPar ville:")
    if df['ville'].notna().any():
        print(df['ville'].value_counts())
    
    print(f"\nPrix (statistiques):")
    if df['prix'].notna().any():
        print(df['prix'].describe())
    
    print(f"\nSurface (statistiques):")
    if df['surface_m2'].notna().any():
        print(df['surface_m2'].describe())
    
    print(f"\nTaux de complétude des champs:")
    for col in COLUMNS:
        completeness = df[col].notna().sum() / len(df) * 100
        print(f"  {col:20s}: {completeness:5.1f}%")
    
    # Aperçu des données
    print(f"\n\nAperçu des données (5 premières lignes):")
    print(df.head().to_string())
    
else:
    print("⚠ Aucune donnée à exporter")
